# Etapa 1 — Limpieza, estandarización y definición de Mc

**Entregables:**
- `seismos_clean.parquet` — dataset limpio listo para EDA y modelado
- Tabla de Mc por época
- Reporte de registros descartados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path

## 1. Detección de encoding y carga

In [ ]:
# Inspección manual de bytes para confirmar encoding
with open('SSNMX.csv', 'rb') as f:
    muestra = f.read(500)
print(muestra)

In [ ]:
ENCODING = 'latin-1'  # Ajustar si la celda anterior muestra otra cosa

df_raw = pd.read_csv('SSNMX.csv', encoding=ENCODING, low_memory=False)

print(f'Filas cargadas: {len(df_raw):,}')
print(f'Columnas: {list(df_raw.columns)}')
df_raw.head(3)

## 2. Reporte de calidad inicial

In [ ]:
descartes = {}  # Acumulador del reporte de descartes

def snapshot(label, df):
    print(f'[{label}] → {len(df):,} registros')

snapshot('raw', df_raw)

## 3. Tipado: Magnitud y Profundidad a numérico

In [ ]:
df = df_raw.copy()

df['Magnitud']    = pd.to_numeric(df['Magnitud'],    errors='coerce')
df['Profundidad'] = pd.to_numeric(df['Profundidad'], errors='coerce')

mag_no_num = df['Magnitud'].isna().sum()
descartes['magnitud_no_convertible'] = int(mag_no_num)
print(f'Registros con magnitud no convertible: {mag_no_num}')

df = df.dropna(subset=['Magnitud'])
snapshot('post-tipado', df)

## 4. Construcción del timestamp UTC canónico

In [ ]:
# Combinar Fecha UTC + Hora UTC en un solo datetime timezone-aware
df['datetime_utc'] = pd.to_datetime(
    df['Fecha UTC'] + ' ' + df['Hora UTC'],
    format='%d/%m/%Y %H:%M:%S',
    errors='coerce',
    utc=True
)

# Hora local para el filtro "hora del día" en el dashboard
df['hora_local'] = pd.to_datetime(
    df['Fecha'] + ' ' + df['Hora'],
    format='%d/%m/%Y %H:%M:%S',
    errors='coerce'
).dt.hour

ts_nulos = df['datetime_utc'].isna().sum()
descartes['timestamp_invalido'] = int(ts_nulos)
print(f'Timestamps no convertibles: {ts_nulos}')

df = df.dropna(subset=['datetime_utc'])
snapshot('post-datetime', df)

## 5. Limpieza de texto y extracción de estado

In [ ]:
df['Referencia de localizacion'] = df['Referencia de localizacion'].str.strip()

# Extraer la abreviatura del estado (último token de 2-4 letras mayúsculas al final)
df['estado'] = (
    df['Referencia de localizacion']
    .str.extract(r',\s*([A-Z]{2,4})\s*$')[0]
)

print('Estados encontrados:')
print(df['estado'].value_counts().head(20))
print(f'\nRegistros sin estado extraído: {df["estado"].isna().sum():,}')

## 6. Filtro geográfico — acotar a México

In [ ]:
LAT_MIN, LAT_MAX =  13.5,  33.5
LON_MIN, LON_MAX = -119.0, -85.5

fuera = ~(
    df['Latitud'].between(LAT_MIN, LAT_MAX) &
    df['Longitud'].between(LON_MIN, LON_MAX)
)
descartes['fuera_de_mexico'] = int(fuera.sum())
print(f'Registros fuera del bounding box México: {fuera.sum()}')

df = df[~fuera]
snapshot('post-filtro-geo', df)

## 7. Duplicados y columnas a descartar

In [ ]:
dupes = df.duplicated().sum()
descartes['duplicados'] = int(dupes)
print(f'Duplicados exactos: {dupes}')

df = df.drop_duplicates()

# Descartar columnas redundantes o de proceso
# Estatus es indistinto entre revisado/verificado → se descarta
# Fecha/Hora local quedan como hora_local (numérica); Fecha UTC / Hora UTC ya están en datetime_utc
cols_a_drop = ['Fecha', 'Hora', 'Fecha UTC', 'Hora UTC', 'Estatus']
df = df.drop(columns=cols_a_drop)

print(f'Columnas finales: {list(df.columns)}')
snapshot('post-dedup', df)

## 8. Reporte de descartes

In [ ]:
total_raw = len(df_raw)
total_clean = len(df)

print('=== REPORTE DE DESCARTES ===')
for motivo, n in descartes.items():
    print(f'  {motivo:<35} {n:>6,}  ({n/total_raw*100:.2f}%)')
print('-' * 50)
print(f'  Registros originales:            {total_raw:>6,}')
print(f'  Registros limpios:               {total_clean:>6,}')
print(f'  Total descartado:                {total_raw - total_clean:>6,}  ({(total_raw-total_clean)/total_raw*100:.2f}%)')

## 9. Magnitud de completitud (Mc) variable por época

Se usa el **método de máxima curvatura**: la Mc es el pico de la distribución
de frecuencias (histograma) de magnitudes, que corresponde al punto de inflexión
de la curva acumulada de Gutenberg-Richter.  
Se aplica en **ventanas de 2 años** a lo largo del catálogo para capturar la
mejora progresiva de la red.

In [ ]:
def mc_maxima_curvatura(magnitudes, bin_size=0.1):
    """Mc = valor de magnitud en el pico del histograma de frecuencias."""
    mag = magnitudes.dropna()
    bins = np.arange(mag.min(), mag.max() + bin_size, bin_size)
    counts, edges = np.histogram(mag, bins=bins)
    pico_idx = np.argmax(counts)
    return round(edges[pico_idx], 1)


VENTANA_AÑOS = 2
df['year'] = df['datetime_utc'].dt.year

años = sorted(df['year'].unique())
epocas = []

for inicio in range(años[0], años[-1] + 1, VENTANA_AÑOS):
    fin = inicio + VENTANA_AÑOS - 1
    mask = df['year'].between(inicio, fin)
    sub = df.loc[mask, 'Magnitud']
    if len(sub) < 100:  # ventana con muy pocos datos (p.ej. 2005 parcial)
        mc = None
    else:
        mc = mc_maxima_curvatura(sub)
    epocas.append({'epoca_inicio': inicio, 'epoca_fin': fin, 'n_sismos': len(sub), 'Mc': mc})

mc_tabla = pd.DataFrame(epocas)
print(mc_tabla.to_string(index=False))

In [ ]:
# Visualización de Mc por época sobre la distribución G-R global
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Izquierda: Mc por época
mc_validas = mc_tabla.dropna(subset=['Mc'])
axes[0].bar(mc_validas['epoca_inicio'].astype(str), mc_validas['Mc'], color='steelblue')
axes[0].set_xlabel('Época (año inicio)')
axes[0].set_ylabel('Mc estimada')
axes[0].set_title('Magnitud de Completitud (Mc) por época')
axes[0].tick_params(axis='x', rotation=45)
axes[0].axhline(mc_validas['Mc'].mean(), color='red', linestyle='--', label=f'Media = {mc_validas["Mc"].mean():.1f}')
axes[0].legend()

# Derecha: distribución acumulada G-R global
mags = df['Magnitud'].dropna()
bin_size = 0.1
bins = np.arange(mags.min(), mags.max() + bin_size, bin_size)
counts, edges = np.histogram(mags, bins=bins)
cum_counts = counts[::-1].cumsum()[::-1]  # frecuencia acumulada descendente

axes[1].semilogy(edges[:-1], cum_counts, 'o', markersize=3, color='gray')
axes[1].set_xlabel('Magnitud')
axes[1].set_ylabel('N (acumulado, escala log)')
axes[1].set_title('Curva de Gutenberg-Richter (catálogo completo)')
if not mc_validas.empty:
    mc_media = mc_validas['Mc'].mean()
    axes[1].axvline(mc_media, color='red', linestyle='--', label=f'Mc media = {mc_media:.1f}')
    axes[1].legend()

plt.tight_layout()
plt.savefig('mc_por_epoca.png', dpi=150)
plt.show()
print('Figura guardada: mc_por_epoca.png')

## 10. Asignar Mc a cada registro y añadir columna de época

In [ ]:
# Mapeo año → Mc de su época
year_to_mc = {}
for _, row in mc_tabla.iterrows():
    for y in range(int(row['epoca_inicio']), int(row['epoca_fin']) + 1):
        year_to_mc[y] = row['Mc']

df['Mc_epoca'] = df['year'].map(year_to_mc)

# Columna booleana: ¿está este sismo por encima de la Mc de su época?
df['sobre_Mc'] = df['Magnitud'] >= df['Mc_epoca']

print(f'Sismos sobre Mc de su época: {df["sobre_Mc"].sum():,}')
print(f'Sismos bajo Mc (microsismos para análisis estadístico): {(~df["sobre_Mc"]).sum():,}')

## 11. Guardar entregables

In [ ]:
# Dataset limpio completo (incluye todos los sismos con columna sobre_Mc)
df.to_parquet('seismos_clean.parquet', index=False)
print(f'Guardado: seismos_clean.parquet  ({Path("seismos_clean.parquet").stat().st_size / 1e6:.1f} MB)')

# Tabla de Mc por época
mc_tabla.to_csv('mc_por_epoca.csv', index=False)
print(f'Guardado: mc_por_epoca.csv')

print('\n=== Etapa 1 completada ===')